# Deploy the U-Net wildfire model to Google Cloud (project `wildfirepinn`)

This notebook puts the trained U-Net models in the cloud next to the PINN model, in the same project. When it is done,
the model shows up in **Agent Platform → Models → Model Registry** as `wildfire-unet`, and the backend can call it.

What gets deployed: the fire check, the current-fire layer, the hourly model (+1, +3, +6 hours) and the daily model
(+24 hours). It uses the models already trained and saved on Drive; nothing is retrained here.

Before you start:

- Your Google account needs access to `wildfirepinn`: Vertex AI User, Cloud Build Editor, Artifact Registry
  Administrator, Storage Admin and Service Account User (or Editor). The project owner can add these with
  **Grant access** on the Cloud Hub page.
- Check the region of the PINN model in Model Registry and put the same one in `GCP_REGION` below.
- A deployed model costs about $2-3 a day while it is deployed. Undeploy it when you are not using it (step 7).

Run order: steps 1-4 first (nothing in the cloud yet), then set `DEPLOY = True` and run step 5.

## 1. Settings

In [ ]:
import os, sys, json, re, subprocess, shutil, glob, time, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_ROOT    = "/content/drive/MyDrive/Colab Notebooks/CMPE 295 Master's Project/next-day-wildfire-spread"
BUNDLE_DIR   = os.path.join(DATA_ROOT, 'unet_bundle_v3')       # trained models: daily files plus goes_h1, goes_h3, goes_h6
STATIC_CACHE = os.path.join(DATA_ROOT, 'static_ca_1km.npz')    # terrain and NDVI for California (built in step 3 if missing)
SERVING_DIR  = '/content/serving' if IN_COLAB else '/tmp/serving'
ENDPOINT_FILE = os.path.join(DATA_ROOT, 'vertex_endpoint.json')  # the endpoint ID is saved here after deploying

GCP_PROJECT   = 'wildfirepinn'                  # same project as the PINN model
GCP_REGION    = 'us-central1'                   # use the region shown next to the PINN model in Model Registry
EE_PROJECT    = 'wildfire-prediction-spread'    # Earth Engine project, only used to build the terrain file
AR_REPO       = 'wildfire'
MODEL_NAME    = 'wildfire-unet'
ENDPOINT_NAME = 'wildfire-unet-endpoint'
IMAGE_BASE    = f'{GCP_REGION}-docker.pkg.dev/{GCP_PROJECT}/{AR_REPO}/{MODEL_NAME}'
MACHINE_TYPE  = 'n1-standard-2'

BUILD_PACKAGE = True     # step 3
LOCAL_TEST    = True     # step 4
DEPLOY        = False    # step 5: builds and deploys in the cloud (costs money while deployed)

os.makedirs(SERVING_DIR, exist_ok=True)


def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ''
    except Exception:
        return os.environ.get(name, '')


FIRMS_MAP_KEY = _secret('FIRMS_MAP_KEY')    # optional: only the daily model needs it (Colab Secrets, key icon)

## 2. Sign in

In [ ]:
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()          # lets the gcloud commands below act as you


def sh(cmd, check=True):
    """Runs a command and prints the end of its output."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-1500:])
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')
    return r.stdout.strip()

## 3. Build the package

The package is a folder with everything the cloud needs:

- `serve.py`: the web server (local: `python serve.py --port 8081`; Vertex AI starts it through the Dockerfile)
- `inference.py`: loads and runs the models, answers requests (the backend contract lives here)
- `native_ingestion.py`: fetches what the models need (GOES-18 on AWS, Open-Meteo weather, terrain file) and the fire check
- `postprocessing.py`: grid ids, cell polygons, one GeoJSON with horizons 0/1/3/6, clipping to the requested bbox
- `settings.py`: shared settings
- `requirements.txt` and `Dockerfile`: how to build the container
- `models/`: the trained models from Drive
- `static_ca_1km.npz`: terrain and NDVI for California, so the hourly model doesn't need Earth Engine in the cloud

You don't need to edit the code cells below; they are the same code the backend runs locally on port 8081.

In [ ]:
%%writefile {SERVING_DIR}/settings.py
"""Settings shared by all modules: grid, GOES-18, model files, and the contract defaults."""
import os, json, math, re, glob
import numpy as np, pandas as pd, requests, joblib
import xml.etree.ElementTree as ET
import xarray as xr
import tensorflow as tf
from dataclasses import dataclass, field
from pyproj import Transformer, CRS
from scipy import ndimage
from sklearn.cluster import DBSCAN
import threading, time as _time
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache


GRID_CRS = 'EPSG:3310'
RES_M = 1000.0
LATTICE_X0 = -430000.0
LATTICE_Y0 = 480000.0
LATTICE_NCOLS = 1000
CA_BBOX = (-124.5, 32.5, -114.0, 42.1)
GOES_TILE = 96
DAILY_TILE = 64
GOES_BUCKET = 'noaa-goes18'
GOES_PRODUCT = 'ABI-L2-FDCC'
GOES_FILE_STRIDE = 2
GOES_HORIZONS_H = (1, 3, 6)
P_EMIT = 0.05
P_MAX = 0.6
W_UNET = 0.5
WEATHER_GRID_N = 9
FIRE_CODES = [10, 30, 11, 31, 12, 32, 13, 33, 14, 34, 15, 35]
FIRE_CONF = [1.0, 1.0, 0.9, 0.9, 0.8, 0.8, 0.5, 0.5, 0.3, 0.3, 0.1, 0.1]
FIRE_THRESHOLD = 0.5
FIRE_CHECK_RADIUS_KM = 50
GOES_CHANNELS = ['fire_now', 'fire_prev3h', 'fire_24h', 'tmp_c', 'rh', 'wind_u', 'wind_v', 'gust', 'precip', 'elevation', 'slope', 'NDVI']
GOES_FIRE_CHANNELS = 3
OPEN_METEO_VARS = ['temperature_2m', 'dew_point_2m', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'precipitation']
RAW_INPUTS = ['elevation', 'th', 'vs', 'tmmn', 'tmmx', 'sph', 'pr', 'pdsi', 'NDVI', 'population', 'erc']
FEATURES = ['elevation', 'vs', 'wind_u', 'wind_v', 'tmmn', 'tmmx', 'sph', 'pr', 'pdsi', 'NDVI', 'population', 'erc', 'PrevFireMask']
UNNORMALIZED = {'PrevFireMask'}
VALID_RANGE = {'elevation': (-100.0, 4500.0), 'th': (0.0, 360.0), 'vs': (0.0, 30.0), 'tmmn': (230.0, 330.0), 'tmmx': (230.0, 330.0), 'sph': (0.0, 0.05), 'pr': (0.0, 300.0), 'pdsi': (-15.0, 15.0), 'NDVI': (-10000.0, 10000.0), 'population': (0.0, 100000.0), 'erc': (0.0, 150.0)}
FIRMS_ARCHIVE_SOURCES = ['VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP']
FIRMS_NRT_SOURCES = ['VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT', 'VIIRS_NOAA21_NRT']
PINN_CFG = {'s_m': 1000.0, 'inside_negative': True}   # PINN not used in this service


BUNDLE_DIR = os.environ.get('WILDFIRE_MODEL_DIR', '/app/models')
STATIC_CACHE = os.environ.get('WILDFIRE_STATIC_CACHE', '/app/static_ca_1km.npz')
GOES_CACHE = os.environ.get('WILDFIRE_GOES_CACHE', '/tmp/goes_cache')
os.makedirs(GOES_CACHE, exist_ok=True)
FIRMS_MAP_KEY = os.environ.get('FIRMS_MAP_KEY', '')
_to_grid = Transformer.from_crs('EPSG:4326', GRID_CRS, always_xy=True)
_to_wgs = Transformer.from_crs(GRID_CRS, 'EPSG:4326', always_xy=True)
_CONF_LUT = np.zeros(256, np.float32)
for _code, _conf in zip(FIRE_CODES, FIRE_CONF):
    _CONF_LUT[_code] = _conf
_S3_NS = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
_GOES_INDEX_CACHE, _HOURLY, _DAILY, _STATIC = {}, {}, {}, {}
PHI, PINN_BOUNDS = None, None
EE_READY = False
try:
    import ee, google.auth
    _creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/earthengine',
                                           'https://www.googleapis.com/auth/cloud-platform'])
    ee.Initialize(_creds, project=os.environ.get('GCP_PROJECT'))
    EE_READY = True
except Exception as _ex:
    print('Earth Engine not available, daily model is off:', _ex)


# contract with the backend (ML adapter)
MODEL_VERSION = 'unet-goes-hourly-1.0'
CONTRACT_HORIZONS = (0, 1, 3, 6)      # 0 = current fire (GOES-18, last hour)
DEFAULT_HOUR_UTC = 21                 # date-only requests use 21:00 UTC (peak afternoon burn window)
MIN_CONFIDENCE = 0.01                 # lowest confidence_threshold accepted (keeps responses small)
MAX_BBOX_KM = 500                     # largest bbox side the service will run
FIRE_SEARCH_PAD_KM = 25               # fires up to this far outside the bbox still count


__all__ = [n for n in list(globals()) if not n.startswith('__')]   # lets the other modules import every name

In [ ]:
%%writefile {SERVING_DIR}/postprocessing.py
"""Model grid -> GeoJSON: grid ids, cell polygons, risk labels, one FeatureCollection for all horizons, bbox clipping."""
import math
import numpy as np, pandas as pd
from dataclasses import dataclass
from settings import *


@dataclass(frozen=True)
class Grid:
    x0: float   # west edge (m)
    y0: float   # north edge (m)
    h: int
    w: int
    res: float = RES_M

    @classmethod
    def around_xy(cls, x, y, size=GOES_TILE, res=RES_M):
        c0 = round((x - size * res / 2 - LATTICE_X0) / res)
        r0 = round((LATTICE_Y0 - (y + size * res / 2)) / res)
        return cls(LATTICE_X0 + c0 * res, LATTICE_Y0 - r0 * res, size, size, res)

    @classmethod
    def around(cls, lat, lon, size=GOES_TILE, res=RES_M):
        return cls.around_xy(*_to_grid.transform(lon, lat), size=size, res=res)

    @property
    def bounds(self):
        return self.x0, self.x0 + self.w * self.res, self.y0 - self.h * self.res, self.y0

    def centers(self):
        xs = self.x0 + (np.arange(self.w) + 0.5) * self.res
        ys = self.y0 - (np.arange(self.h) + 0.5) * self.res
        return np.meshgrid(xs, ys)

    def grid_ids(self):
        ncols = int(round(LATTICE_NCOLS * RES_M / self.res))      # 1000 for 1 km cells, 4000 for 250 m cells
        c0 = round((self.x0 - LATTICE_X0) / self.res)
        r0 = round((LATTICE_Y0 - self.y0) / self.res)
        r, c = np.mgrid[r0:r0 + self.h, c0:c0 + self.w]
        return (r * ncols + c).astype(np.int64)


def grid_id_to_center_lonlat(gid, res=RES_M):
    r, c = divmod(int(gid), LATTICE_NCOLS)
    return _to_wgs.transform(LATTICE_X0 + (c + 0.5) * res, LATTICE_Y0 - (r + 0.5) * res)


def risk_label(p, thr):
    return np.where(p >= thr['HIGH_RISK'], 'HIGH_RISK',
                    np.where(p >= thr['MODERATE_RISK'], 'MODERATE_RISK', 'LOW_RISK'))


def to_geojson(p, grid, timestamp, thr, p_emit=P_EMIT, burned=None, extra_layers=None, collection_props=None, ndp=6):
    p = np.asarray(p, float)
    burned = np.zeros(p.shape, bool) if burned is None else np.asarray(burned, bool)
    ids, labels = grid.grid_ids(), risk_label(np.nan_to_num(p), thr)
    cx = grid.x0 + np.arange(grid.w + 1) * grid.res
    cy = grid.y0 - np.arange(grid.h + 1) * grid.res
    LON, LAT = _to_wgs.transform(*np.meshgrid(cx, cy))
    LON, LAT = np.round(LON, ndp), np.round(LAT, ndp)
    feats = []
    rows, cols = np.where((np.isfinite(p) & (p >= p_emit)) | burned)
    for r, c in zip(rows.tolist(), cols.tolist()):
        # SW, SE, NE, NW, SW
        ring = [[LON[r + 1, c], LAT[r + 1, c]], [LON[r + 1, c + 1], LAT[r + 1, c + 1]],
                [LON[r, c + 1], LAT[r, c + 1]], [LON[r, c], LAT[r, c]], [LON[r + 1, c], LAT[r + 1, c]]]
        if burned[r, c]:
            props = {'grid_id': int(ids[r, c]), 'fire_probability': 1.0, 'risk_label': 'ACTIVE_FIRE'}
        else:
            props = {'grid_id': int(ids[r, c]), 'fire_probability': round(float(p[r, c]), 3),
                     'risk_label': str(labels[r, c])}
        for name, layer in (extra_layers or {}).items():
            if layer is not None:
                v = layer[r, c]
                props[name] = None if not np.isfinite(v) else round(float(v), 3)
        feats.append({'type': 'Feature', 'properties': props,
                      'geometry': {'type': 'Polygon', 'coordinates': [[[float(a), float(b)] for a, b in ring]]}})
    fc = {'type': 'FeatureCollection', 'timestamp': timestamp}
    fc.update(collection_props or {})
    fc['features'] = feats
    return fc


def validate_feature_collection(fc):
    ids = [f['properties']['grid_id'] for f in fc['features']]
    assert len(ids) == len(set(ids)), 'duplicate grid_id'
    for f in fc['features']:
        ring = f['geometry']['coordinates'][0]
        assert len(ring) == 5 and ring[0] == ring[-1], 'ring not closed'
        assert sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:])) > 0, 'ring not counter-clockwise'
        pr = f['properties']
        assert 0 <= pr['fire_probability'] <= 1
        assert pr['risk_label'] in {'LOW_RISK', 'MODERATE_RISK', 'HIGH_RISK', 'ACTIVE_FIRE'}
    json.dumps(fc)


def grid_id_to_polygon(gid, res=RES_M):
    """Full (unclipped) [lon, lat] ring of a 1 km lattice cell, counter-clockwise, closed."""
    r, c = divmod(int(gid), LATTICE_NCOLS)
    xs = [LATTICE_X0 + c * res, LATTICE_X0 + (c + 1) * res]
    ys = [LATTICE_Y0 - (r + 1) * res, LATTICE_Y0 - r * res]
    lon, lat = _to_wgs.transform([xs[0], xs[1], xs[1], xs[0], xs[0]], [ys[0], ys[0], ys[1], ys[1], ys[0]])
    return [[round(a, 6), round(b, 6)] for a, b in zip(lon, lat)]


def model_grid_for_bbox(bbox, min_km=GOES_TILE, max_km=MAX_BBOX_KM):
    """Square model grid centred on the bbox: at least min_km wide (the model needs context), a multiple of 16 cells."""
    w, s, e, n = bbox
    xs, ys = _to_grid.transform([w, e, e, w], [s, s, n, n])
    span_km = max(max(xs) - min(xs), max(ys) - min(ys)) / 1000.0
    if span_km > max_km:
        raise ValueError(f'bbox is {span_km:.0f} km across; the largest supported is {max_km} km')
    size = max(min_km, int(math.ceil((span_km + 2) / 16.0)) * 16)
    return Grid.around_xy((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, size=size)


def _cross_x(a, b, x):
    t = (x - a[0]) / (b[0] - a[0])
    return [x, a[1] + t * (b[1] - a[1])]


def _cross_y(a, b, y):
    t = (y - a[1]) / (b[1] - a[1])
    return [a[0] + t * (b[0] - a[0]), y]


def _ring_area(pts):
    return 0.5 * sum(pts[i - 1][0] * pts[i][1] - pts[i][0] * pts[i - 1][1] for i in range(len(pts)))


def clip_ring_to_bbox(ring, bbox, ndp=5):
    """Cuts a closed [lon, lat] ring to the bbox (keeps it a single Polygon). Returns a closed ring, or None if nothing is left."""
    w, s, e, n = bbox
    pts = [list(p) for p in ring[:-1]]
    if all(w <= x <= e and s <= y <= n for x, y in pts):
        out = pts
    else:
        edges = ((lambda p: p[0] >= w, lambda a, b: _cross_x(a, b, w)), (lambda p: p[0] <= e, lambda a, b: _cross_x(a, b, e)),
                 (lambda p: p[1] >= s, lambda a, b: _cross_y(a, b, s)), (lambda p: p[1] <= n, lambda a, b: _cross_y(a, b, n)))
        out = pts
        for inside, cross in edges:
            nxt = []
            for i, cur in enumerate(out):
                prev = out[i - 1]
                if inside(cur):
                    if not inside(prev):
                        nxt.append(cross(prev, cur))
                    nxt.append(cur)
                elif inside(prev):
                    nxt.append(cross(prev, cur))
            out = nxt
            if not out:
                return None
    rounded = []
    for x, y in out:
        p = [round(min(max(x, w), e), ndp), round(min(max(y, s), n), ndp)]
        if not rounded or p != rounded[-1]:
            rounded.append(p)
    if len(rounded) > 1 and rounded[0] == rounded[-1]:
        rounded.pop()
    if len(rounded) < 3 or abs(_ring_area(rounded)) < 1e-10:
        return None
    return rounded + [rounded[0]]


def unified_feature_collection(layers, bbox, issue_time, p_min=P_EMIT, source_model='unet', status='ok', extra_metadata=None):
    """One FeatureCollection for all horizons. layers: [{'h', 'p', 'grid', 'thresholds', 'burned'}]."""
    fmt = '%Y-%m-%dT%H:%M:%SZ'
    feats = []
    for lay in layers:
        fc = to_geojson(lay['p'], lay['grid'], '', lay['thresholds'], p_emit=p_min, burned=lay['burned'])
        for f in fc['features']:
            ring = clip_ring_to_bbox(f['geometry']['coordinates'][0], bbox)
            if ring is None:
                continue
            props = f['properties']
            props.update({'horizon_hours': int(lay['h']), 'source_model': source_model, 'w_unet': 1.0, 'members': 1})
            feats.append({'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [ring]}, 'properties': props})
    meta = {'timestamp': pd.Timestamp.now('UTC').strftime(fmt), 'issue_time': pd.Timestamp(issue_time).strftime(fmt),
            'source_model': source_model, 'model_version': MODEL_VERSION, 'grid_resolution_km': RES_M / 1000.0,
            'total_cells': len(feats), 'status': status, 'bbox': [float(v) for v in bbox],
            'horizons': sorted({int(l['h']) for l in layers}),
            'risk_thresholds': {str(l['h']): {k: round(float(v), 4) for k, v in l['thresholds'].items()}
                                for l in layers if l['h'] > 0}}
    meta.update(extra_metadata or {})
    return {'type': 'FeatureCollection', 'metadata': meta, 'features': feats}


__all__ = [n for n in list(globals()) if not n.startswith('__')]   # lets the other modules import every name

In [ ]:
%%writefile {SERVING_DIR}/native_ingestion.py
"""Native ingestion: everything the models read for a place and time (GOES-18 on AWS, Open-Meteo, terrain file,
and for the daily model Earth Engine + FIRMS). Also the active-fire check."""
import os, re, glob, math, threading, time as _time
import numpy as np, pandas as pd, requests
import xml.etree.ElementTree as ET
import xarray as xr
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from pyproj import Transformer, CRS
from sklearn.cluster import DBSCAN
from settings import *
from postprocessing import Grid

_NC_LOCK = threading.Lock()   # the NetCDF/HDF5 readers are not thread-safe


def _utc(t):
    t = pd.Timestamp(t)
    return t.tz_localize('UTC') if t.tzinfo is None else t.tz_convert('UTC')


def mask_to_conf(mask):
    m = np.clip(np.nan_to_num(np.asarray(mask, float), nan=0.0), 0, 255).astype(int)
    return _CONF_LUT[m]


def _s3_list_keys(bucket, prefix):
    keys, token = [], None
    while True:
        params = {'list-type': '2', 'prefix': prefix}
        if token:
            params['continuation-token'] = token
        r = requests.get(f'https://{bucket}.s3.amazonaws.com/', params=params, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        keys += [e.text for e in root.findall('s3:Contents/s3:Key', _S3_NS)]
        if root.findtext('s3:IsTruncated', default='false', namespaces=_S3_NS) != 'true':
            return keys
        token = root.findtext('s3:NextContinuationToken', namespaces=_S3_NS)


def _s3_download(bucket, key):
    dest = os.path.join(GOES_CACHE, os.path.basename(key))
    if not os.path.exists(dest):
        r = requests.get(f'https://{bucket}.s3.amazonaws.com/{key}', timeout=120)
        r.raise_for_status()
        with open(dest + '.part', 'wb') as f:
            f.write(r.content)
        os.replace(dest + '.part', dest)
    return dest


def goes_keys_for_hour(t_hour):
    t = _utc(t_hour)
    prefix = f'{GOES_PRODUCT}/{t:%Y}/{t.dayofyear:03d}/{t:%H}/'
    return sorted(k for k in _s3_list_keys(GOES_BUCKET, prefix) if k.endswith('.nc'))[::GOES_FILE_STRIDE]


def _goes_indices(ds, grid):
    # row/col of the GOES pixel under each grid cell centre, cached per grid and file geometry
    p = ds['goes_imager_projection'].attrs
    h, lon0 = float(p['perspective_point_height']), float(p['longitude_of_projection_origin'])
    x, y = ds['x'].values.astype(float), ds['y'].values.astype(float)    # scan angles in radians
    key = (grid, h, lon0, x[0], x[-1], len(x), y[0], y[-1], len(y))
    if key not in _GOES_INDEX_CACHE:
        geos = CRS.from_proj4(f"+proj=geos +h={h} +lon_0={lon0} +sweep={p.get('sweep_angle_axis', 'x')} "
                              f"+a={float(p['semi_major_axis'])} +b={float(p['semi_minor_axis'])} +units=m +no_defs")
        X, Y = grid.centers()
        gx, gy = Transformer.from_crs(GRID_CRS, geos, always_xy=True).transform(X, Y)
        ix = np.rint((np.asarray(gx) / h - x[0]) / (x[1] - x[0]))
        iy = np.rint((np.asarray(gy) / h - y[0]) / (y[1] - y[0]))
        ok = np.isfinite(ix) & np.isfinite(iy) & (ix >= 0) & (ix < len(x)) & (iy >= 0) & (iy < len(y))
        _GOES_INDEX_CACHE[key] = (np.where(ok, iy, 0).astype(int), np.where(ok, ix, 0).astype(int), ok)
    return _GOES_INDEX_CACHE[key]


def _window(iy, ix, ok):
    """Smallest block of GOES rows/columns that covers the grid, so only that part of each file is read."""
    if not ok.any():
        return 0, 1, 0, 1
    return int(iy[ok].min()), int(iy[ok].max()) + 1, int(ix[ok].min()), int(ix[ok].max()) + 1


@lru_cache(maxsize=4096)
def read_fdc_on_grid(path, grid):
    """One GOES file -> fire confidence, clear-sky look (0/1) and fire power (MW) on the grid (cached per file and grid)."""
    import xarray as xr
    with _NC_LOCK, xr.open_dataset(path) as ds:
        iy, ix, ok = _goes_indices(ds, grid)
        r0, r1, c0, c1 = _window(iy, ix, ok)
        part = {v: ds[v].isel(y=slice(r0, r1), x=slice(c0, c1)).values for v in ('Mask', 'DQF', 'Power')}
    jy, jx = np.clip(iy - r0, 0, r1 - r0 - 1), np.clip(ix - c0, 0, c1 - c0 - 1)
    conf = np.where(ok, mask_to_conf(part['Mask'][jy, jx]), 0.0)
    d = np.asarray(part['DQF'][jy, jx], float)
    clear = ok & np.isfinite(d) & (d <= 1)     # DQF 0 = good fire pixel, 1 = good fire-free land
    frp = np.where(ok, np.nan_to_num(np.asarray(part['Power'][jy, jx], float), nan=0.0), 0.0)
    return conf.astype(np.float32), clear.astype(np.float32), frp.astype(np.float32)


def _download_all(keys, workers=16):
    """Downloads many GOES files at the same time; returns {key: local path}."""
    with ThreadPoolExecutor(workers) as pool:
        return dict(zip(keys, pool.map(lambda k: _s3_download(GOES_BUCKET, k), keys)))


def goes_aws_hourly(grid, t0, n_hours):
    """Hourly max fire confidence, number of clear looks and max fire power, each [h, w, n_hours]."""
    t0 = _utc(t0).floor('h')
    hours = [t0 + pd.Timedelta(hours=k) for k in range(n_hours)]
    with ThreadPoolExecutor(16) as pool:
        keys_per_hour = list(pool.map(goes_keys_for_hour, hours))
    paths = _download_all([k for ks in keys_per_hour for k in ks])
    conf = np.zeros((grid.h, grid.w, n_hours), np.float32)
    nval, frp = np.zeros_like(conf), np.zeros_like(conf)
    for k, keys in enumerate(keys_per_hour):
        if not keys:
            print(f'  no GOES files for {hours[k]:%Y-%m-%d %H}Z')
        for key in keys:
            c, v, f = read_fdc_on_grid(paths[key], grid)
            conf[..., k] = np.maximum(conf[..., k], c)
            nval[..., k] += v
            frp[..., k] = np.maximum(frp[..., k], f)
    return conf, nval, frp


def _geos_crs(p):
    return CRS.from_proj4(f"+proj=geos +h={float(p['perspective_point_height'])} "
                          f"+lon_0={float(p['longitude_of_projection_origin'])} +sweep={p.get('sweep_angle_axis', 'x')} "
                          f"+a={float(p['semi_major_axis'])} +b={float(p['semi_minor_axis'])} +units=m +no_defs")


def goes_fire_points(t_end, hours=1, bbox=CA_BBOX):
    """Every GOES-18 fire pixel (confidence 0.5 or more) inside bbox in the hours before t_end."""
    t_end = _utc(t_end).floor('h')
    w, s, e, n = bbox
    frames = []
    for k in range(hours, 0, -1):
        for key in goes_keys_for_hour(t_end - pd.Timedelta(hours=k)):
            path = _s3_download(GOES_BUCKET, key)
            with _NC_LOCK, xr.open_dataset(path) as ds:
                attrs = dict(ds['goes_imager_projection'].attrs)
                conf = mask_to_conf(ds['Mask'].values)
                iy, ix = np.where(conf >= FIRE_THRESHOLD)
                if len(iy) == 0:
                    continue
                h = float(attrs['perspective_point_height'])
                gx = ds['x'].values[ix].astype(float) * h
                gy = ds['y'].values[iy].astype(float) * h
                frp = np.nan_to_num(np.asarray(ds['Power'].values[iy, ix], float))
                c = conf[iy, ix]
            lon, lat = Transformer.from_crs(_geos_crs(attrs), 'EPSG:4326', always_xy=True).transform(gx, gy)
            keep = (lon >= w) & (lon <= e) & (lat >= s) & (lat <= n)
            frames.append(pd.DataFrame({'lat': lat[keep], 'lon': lon[keep], 'confidence': c[keep],
                                        'frp_mw': frp[keep], 'scan': re.search(r'_s(\d{11})', key).group(1)}))
    if not frames:
        return pd.DataFrame(columns=['lat', 'lon', 'confidence', 'frp_mw', 'scan'])
    return pd.concat(frames, ignore_index=True)


def find_active_fires(points, eps_km=4.0, min_scans=2):
    """Groups fire pixels into fires. A fire has to show up in at least min_scans scans."""
    cols = ['lat', 'lon', 'pixels', 'scans', 'max_frp_mw']
    if len(points) == 0:
        return pd.DataFrame(columns=cols)
    labels = DBSCAN(eps=eps_km / 6371.0, min_samples=1, metric='haversine').fit_predict(
        np.radians(points[['lat', 'lon']].to_numpy(float)))
    fires = (points.assign(fire=labels).groupby('fire')
                   .agg(lat=('lat', 'mean'), lon=('lon', 'mean'), pixels=('lat', 'size'),
                        scans=('scan', 'nunique'), max_frp_mw=('frp_mw', 'max')))
    fires = fires[fires['scans'] >= min_scans].sort_values('max_frp_mw', ascending=False)
    return fires.reset_index(drop=True)[cols]


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1 = math.radians(lat1), math.radians(lon1)
    lat2, lon2 = np.radians(np.asarray(lat2, float)), np.radians(np.asarray(lon2, float))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def bbox_around(lat, lon, radius_km):
    dlat = radius_km / 111.0
    dlon = radius_km / (111.0 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)


def fire_check(lat, lon, t_end, radius_km=FIRE_CHECK_RADIUS_KM, fires=None):
    """Is there an active fire within radius_km of (lat, lon) in the hour before t_end?"""
    if fires is None:
        fires = find_active_fires(goes_fire_points(t_end, bbox=bbox_around(lat, lon, radius_km)))
    if len(fires) == 0:
        return {'active': False, 'nearest_km': None, 'fires': fires}
    d = haversine_km(lat, lon, fires['lat'], fires['lon'])
    near = fires.assign(distance_km=np.round(d, 1))[d <= radius_km].reset_index(drop=True)
    return {'active': len(near) > 0, 'nearest_km': round(float(np.min(d)), 1), 'fires': near}


def relative_humidity(t_c, td_c):
    return np.clip(100.0 * np.exp(17.625 * td_c / (243.04 + td_c)) / np.exp(17.625 * t_c / (243.04 + t_c)), 0, 100)


def window(conf, nval, a, b):
    return conf[..., a:b].max(-1), nval[..., a:b].sum(-1)


def make_label(conf_w, nval_w):
    return np.where(conf_w >= FIRE_THRESHOLD, 1.0, np.where(nval_w > 0, 0.0, -1.0)).astype(np.float32)


def goes_inputs(conf, nval, k, weather, static):
    # k is the issue hour in the stack; hour k-1 is "the last hour"
    now_c, _ = window(conf, nval, k - 1, k)
    prev_c, _ = window(conf, nval, max(k - 4, 0), k - 1)
    day_c, _ = window(conf, nval, max(k - 24, 0), k)
    w = weather
    x = np.stack([(now_c >= FIRE_THRESHOLD).astype(np.float32), prev_c, day_c,
                  w['TMP'], relative_humidity(w['TMP'], w['DPT']), w['UGRD'], w['VGRD'], w['GUST'], w['ACPC01'],
                  static['elevation'], static['slope'], static['NDVI']], -1)
    return x.astype(np.float32)


def _open_meteo_get(url, params):
    r = requests.get(url, params=params, timeout=90)
    r.raise_for_status()
    return r.json()


def openmeteo_weather(grid, t, n=WEATHER_GRID_N):
    from scipy.interpolate import RegularGridInterpolator
    t = _utc(t).floor('h')
    xmin, xmax, ymin, ymax = grid.bounds
    xs, ys = np.linspace(xmin, xmax, n), np.linspace(ymin, ymax, n)
    lon, lat = _to_wgs.transform(*np.meshgrid(xs, ys))
    recent = pd.Timestamp.now('UTC') - t < pd.Timedelta(days=80)
    url = ('https://api.open-meteo.com/v1/forecast' if recent
           else 'https://historical-forecast-api.open-meteo.com/v1/forecast')
    js = _open_meteo_get(url, {'latitude': ','.join(f'{v:.4f}' for v in np.ravel(lat)),
                               'longitude': ','.join(f'{v:.4f}' for v in np.ravel(lon)),
                               'hourly': ','.join(OPEN_METEO_VARS), 'wind_speed_unit': 'ms', 'timezone': 'UTC',
                               'start_date': f'{t:%Y-%m-%d}', 'end_date': f'{t:%Y-%m-%d}'})
    js = js if isinstance(js, list) else [js]
    key = t.strftime('%Y-%m-%dT%H:00')
    v = {name: np.array([loc['hourly'][name][loc['hourly']['time'].index(key)] for loc in js], float).reshape(n, n)
         for name in OPEN_METEO_VARS}
    wd = np.deg2rad(v['wind_direction_10m'])
    fields = {'TMP': v['temperature_2m'], 'DPT': v['dew_point_2m'],
              'UGRD': -v['wind_speed_10m'] * np.sin(wd), 'VGRD': -v['wind_speed_10m'] * np.cos(wd),
              'GUST': v['wind_gusts_10m'], 'ACPC01': v['precipitation']}
    X, Y = grid.centers()
    pts = np.c_[Y.ravel(), X.ravel()]
    return {k: RegularGridInterpolator((ys, xs), np.nan_to_num(f), bounds_error=False, fill_value=None)(pts)
               .reshape(X.shape).astype(np.float32) for k, f in fields.items()}


def static_layers_from_cache(grid, date=None, path=None):
    """Same output as static_layers(), read from the California file instead of Earth Engine."""
    path = path or STATIC_CACHE
    if path not in _STATIC:
        _STATIC[path] = dict(np.load(path))
    s = _STATIC[path]
    full_h, full_w = s['elevation'].shape
    r0 = int(round((float(s['y0']) - grid.y0) / grid.res))
    c0 = int(round((grid.x0 - float(s['x0'])) / grid.res))
    out = {}
    for k in ('elevation', 'slope', 'NDVI'):
        a = np.full((grid.h, grid.w), np.nan, np.float32)
        rs, cs, re_, ce = max(r0, 0), max(c0, 0), min(r0 + grid.h, full_h), min(c0 + grid.w, full_w)
        if rs < re_ and cs < ce:
            a[rs - r0:re_ - r0, cs - c0:ce - c0] = s[k][rs:re_, cs:ce]
        out[k] = a
    return out


static_layers = static_layers_from_cache


def hourly_inputs(grid, t):
    """Native ingestion for the hourly model at time t: GOES-18 fire (last 24 h, AWS), Open-Meteo weather, terrain/NDVI file."""
    t = _utc(t).floor('h')
    conf, nval, _ = goes_aws_hourly(grid, t - pd.Timedelta(hours=24), 24)
    x = goes_inputs(conf, nval, 24, openmeteo_weather(grid, t), static_layers(grid, t))
    burned = window(conf, nval, 0, 24)[0] >= FIRE_THRESHOLD      # seen burning in the last 24 h
    return x, burned


def _ee_latest(collection_id, date, lookback_days, bands):
    d = ee.Date(date)
    col = ee.ImageCollection(collection_id).filterDate(d.advance(-lookback_days, 'day'), d.advance(1, 'day'))
    return ee.Image(col.sort('system:time_start', False).first()).select(bands)


def gee_features(grid, date_str):
    """Daily model inputs (everything except the fire mask) on the grid, in the units used in training."""
    gm = _ee_latest('IDAHO_EPSCOR/GRIDMET', date_str, 5, ['tmmn', 'tmmx', 'vs', 'th', 'sph', 'pr', 'erc'])
    pdsi = _ee_latest('GRIDMET/DROUGHT', date_str, 15, ['pdsi'])
    ndvi = _ee_latest('NASA/VIIRS/002/VNP13A1', date_str, 40, ['NDVI'])
    elev = ee.Image('USGS/SRTMGL1_003').select(['elevation'])
    pop = (ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density')
             .sort('system:time_start', False).first().select(['population_density'], ['population']))
    stack = ee.Image.cat([elev, gm, pdsi, ndvi, pop]).toFloat()
    arr = ee.data.computePixels({
        'expression': stack, 'fileFormat': 'NUMPY_NDARRAY',
        'grid': {'dimensions': {'width': grid.w, 'height': grid.h},
                 'affineTransform': {'scaleX': grid.res, 'shearX': 0, 'translateX': grid.x0,
                                     'shearY': 0, 'scaleY': -grid.res, 'translateY': grid.y0},
                 'crsCode': GRID_CRS}})
    out = {name: np.asarray(arr[name], np.float32) for name in arr.dtype.names}
    out['NDVI'] = out['NDVI'] * 10000.0      # Earth Engine's VNP13A1 is 0-1; training data used NDVI x 10,000
    return out


def _with_datetime(df):
    df = df.copy()
    if 'datetime_utc' in df.columns:
        df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True)
    else:
        t = df['acq_time'].astype(int).astype(str).str.zfill(4)
        df['datetime_utc'] = pd.to_datetime(df['acq_date'].astype(str) + ' ' + t.str[:2] + ':' + t.str[2:], utc=True)
    return df


def drop_non_vegetation(df):
    # FIRMS archive 'type': 0 = vegetation fire, 2 = static land source (refineries, flares), 1/3 = volcano/offshore
    if 'type' in df.columns:
        df = df[pd.to_numeric(df['type'], errors='coerce').fillna(0) == 0]
    return df


def grid_bbox_lonlat(grid, pad_deg=0.05):
    xmin, xmax, ymin, ymax = grid.bounds
    lon, lat = _to_wgs.transform([xmin, xmax, xmax, xmin], [ymin, ymin, ymax, ymax])
    return (min(lon) - pad_deg, min(lat) - pad_deg, max(lon) + pad_deg, max(lat) + pad_deg)


def fetch_firms(start_day, n_days, bbox, sources):
    """VIIRS fire detections from FIRMS in bbox for [start_day, start_day + n_days)."""
    w, s, e, n = bbox
    frames, day = [], pd.Timestamp(start_day)
    end = day + pd.Timedelta(days=n_days)
    while day < end:
        span = min(5, (end - day).days)
        for src in sources:
            url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_MAP_KEY}/{src}/{w},{s},{e},{n}/{span}/{day:%Y-%m-%d}'
            try:
                df = pd.read_csv(url)
                if 'latitude' in df.columns:
                    frames.append(df.assign(source=src))
            except Exception as ex:
                print(f'{src}: {ex}')
        day += pd.Timedelta(days=span)
    if not frames:
        return pd.DataFrame(columns=['latitude', 'longitude', 'datetime_utc'])
    df = _with_datetime(pd.concat(frames, ignore_index=True))
    conf = df['confidence'].astype(str).str.lower().str[0]
    return drop_non_vegetation(df[conf != 'l']).reset_index(drop=True)


def rasterize_points(df, grid):
    mask = np.zeros((grid.h, grid.w), np.float32)
    if df is None or len(df) == 0:
        return mask
    xs, ys = _to_grid.transform(df['longitude'].to_numpy(), df['latitude'].to_numpy())
    col = np.floor((xs - grid.x0) / grid.res).astype(int)
    row = np.floor((grid.y0 - ys) / grid.res).astype(int)
    ok = (row >= 0) & (row < grid.h) & (col >= 0) & (col < grid.w)
    mask[row[ok], col[ok]] = 1.0
    return mask


def utc_day(df, day):
    return df[df['datetime_utc'].dt.strftime('%Y-%m-%d') == day]


def daily_inputs(grid, t):
    """Inputs for a forecast issued at t: fire detections of t's UTC day (up to t) plus Earth Engine layers."""
    t = _utc(t)
    recent = pd.Timestamp.now('UTC') - t < pd.Timedelta(days=60)
    start = (t - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
    dets = fetch_firms(start, 4, grid_bbox_lonlat(grid), FIRMS_NRT_SOURCES if recent else FIRMS_ARCHIVE_SOURCES)
    dets = dets[dets['datetime_utc'] <= t] if len(dets) else dets
    raw = gee_features(grid, t.strftime('%Y-%m-%d'))
    raw['PrevFireMask'] = rasterize_points(utc_day(dets, t.strftime('%Y-%m-%d')) if len(dets) else dets, grid)
    return raw, rasterize_points(dets, grid) > 0


def _keep_goes_cache_warm(every_min=10, hours=25):
    """Keeps the last day of GOES-18 files on the server and deletes files older than two days."""
    while True:
        try:
            now = pd.Timestamp.now('UTC').floor('h')
            hrs = [now - pd.Timedelta(hours=k) for k in range(hours, -1, -1)]
            with ThreadPoolExecutor(16) as pool:
                keys = [k for ks in pool.map(goes_keys_for_hour, hrs) for k in ks]
            _download_all(keys)
            cutoff = _time.time() - 48 * 3600
            for f in glob.glob(os.path.join(GOES_CACHE, '*.nc')):
                if os.path.getmtime(f) < cutoff:
                    os.remove(f)
        except Exception as ex:
            print('GOES warm-up skipped:', str(ex)[:120])
        _time.sleep(every_min * 60)


if os.environ.get('WILDFIRE_WARM_CACHE', '1') == '1':
    threading.Thread(target=_keep_goes_cache_warm, daemon=True).start()


__all__ = [n for n in list(globals()) if not n.startswith('__')]   # lets the other modules import every name

In [ ]:
%%writefile {SERVING_DIR}/inference.py
"""Models: loading, running the hourly (+1/+3/+6 h) and daily (+24 h) U-Nets, and answering requests."""
import os, json
import numpy as np, pandas as pd, joblib
import tensorflow as tf
from dataclasses import dataclass, field
from settings import *
from postprocessing import *
from native_ingestion import *


def current_fire_geojson(grid, t_end, hours=1):
    t_end = _utc(t_end)
    conf, _, frp = goes_aws_hourly(grid, t_end - pd.Timedelta(hours=hours), hours)
    c, f = conf.max(-1), frp.max(-1)
    fmt = '%Y-%m-%dT%H:%M:%SZ'
    return to_geojson(np.zeros_like(c), grid, t_end.strftime(fmt), {'MODERATE_RISK': 1, 'HIGH_RISK': 1},
                      p_emit=2.0, burned=c >= FIRE_THRESHOLD, extra_layers={'fire_confidence': c, 'frp_mw': f},
                      collection_props={'layer': 'current_fire_detection', 'source': f'GOES-18 {GOES_PRODUCT}',
                                        'window_start': (t_end - pd.Timedelta(hours=hours)).strftime(fmt),
                                        'window_end': t_end.strftime(fmt)})


def goes_normalize(X, mu, sd):
    bad = ~(np.isfinite(mu) & np.isfinite(sd))       # bundles trained before the -inf fix have no stats for rain/NDVI
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (np.asarray(X, np.float32) - np.where(bad, 0, mu)) / np.where(bad, 1, sd)
    z[..., bad] = 0.0
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0), -10, 10).astype(np.float32)


def load_goes_bundle(h):
    bdir = os.path.join(BUNDLE_DIR, f'goes_h{h}')
    norm = json.load(open(os.path.join(bdir, 'norm.json')))
    b = {'model': tf.keras.models.load_model(os.path.join(bdir, 'model.keras'), compile=False),
         'mu': np.array(norm['mu'], np.float32), 'sd': np.array(norm['sd'], np.float32),
         'calibrator': joblib.load(os.path.join(bdir, 'calibrator.joblib')),
         'thresholds': json.load(open(os.path.join(bdir, 'thresholds.json'))),
         'manifest': json.load(open(os.path.join(bdir, 'manifest.json')))}
    bad = [GOES_CHANNELS[i] for i in np.where(~(np.isfinite(b['mu']) & np.isfinite(b['sd'])))[0]]
    if bad:
        print(f'+{h} h model was trained without {bad}; retrain them in the final notebook (Section 8), then rebuild this package')
    return b


def fuse(p_unet, p_pinn=None, w_unet=W_UNET):
    p = np.asarray(p_unet, float).copy()
    if p_pinn is not None:
        both = np.isfinite(p_pinn) & np.isfinite(p)
        p[both] = w_unet * p[both] + (1.0 - w_unet) * p_pinn[both]
    return np.clip(p, 0.0, 1.0)


def goes_forecast(grid, t, bundles):
    """Runs the +1/+3/+6 h models. Returns the model inputs, the 'burned in the last 24 h' mask and {h: (p, None, p)}."""
    x, burned = hourly_inputs(grid, t)
    out = {}
    for h, b in bundles.items():
        p = b['model'](goes_normalize(x, b['mu'], b['sd'])[None], training=False).numpy()[0, :, :, 0]
        p = np.minimum(b['calibrator'].predict(p.ravel()).reshape(p.shape), P_MAX)
        out[h] = (p, None, p)
    return x, burned, out


def mask_invalid(fd):
    fd = dict(fd)
    for f in RAW_INPUTS:
        lo, hi = VALID_RANGE[f]
        x = fd[f]
        fd[f] = tf.where(tf.math.is_finite(x) & (x >= lo) & (x <= hi), x, np.nan)
    fd['PrevFireMask'] = tf.where(tf.math.is_finite(fd['PrevFireMask']), fd['PrevFireMask'], -1.0)
    return fd


def add_wind_uv(fd):
    fd = dict(fd)
    th = fd['th'] * (math.pi / 180.0)
    fd['wind_u'] = -fd['vs'] * tf.sin(th)
    fd['wind_v'] = -fd['vs'] * tf.cos(th)
    return fd


def transform_features(fd, features, stats):
    fd = mask_invalid(fd)
    for f in RAW_INPUTS:
        fd[f] = tf.where(tf.math.is_finite(fd[f]), fd[f], stats[f][2])
    fd = add_wind_uv(fd)
    chans = []
    for f in features:
        x = tf.cast(fd[f], tf.float32)
        if f not in UNNORMALIZED:
            lo, hi, mu, sd = stats[f]
            x = (tf.clip_by_value(x, lo, hi) - mu) / sd
        chans.append(tf.where(tf.math.is_finite(x), x, 0.0))
    return tf.stack(chans, axis=-1)


def load_daily_bundle():
    path = os.path.join(BUNDLE_DIR, 'model.keras')
    if not os.path.exists(path):
        return None
    thr = json.load(open(os.path.join(BUNDLE_DIR, 'thresholds.json')))
    thr = dict(thr['validation'] if thr.get('mode') == 'validation' else thr['fixed'])
    thr['MODERATE_RISK'] = max(thr['MODERATE_RISK'], P_EMIT)
    thr['HIGH_RISK'] = min(thr['HIGH_RISK'], P_MAX)
    if thr['MODERATE_RISK'] >= thr['HIGH_RISK']:
        thr['MODERATE_RISK'] = thr['HIGH_RISK'] / 2
    return {'model': tf.keras.models.load_model(path, compile=False),
            'features': json.load(open(os.path.join(BUNDLE_DIR, 'features.json'))),
            'stats': {k: tuple(v) for k, v in json.load(open(os.path.join(BUNDLE_DIR, 'stats.json'))).items()},
            'calibrator': joblib.load(os.path.join(BUNDLE_DIR, 'calibrator.joblib')),
            'thresholds': thr}


def daily_predict(bundle, raw):
    fd = {k: tf.constant(np.asarray(v, np.float32)) for k, v in raw.items()}
    x = transform_features(fd, bundle['features'], bundle['stats'])[None]
    p = bundle['model'](x, training=False).numpy()[0, :, :, 0]
    return np.minimum(bundle['calibrator'].predict(p.ravel()).reshape(p.shape), P_MAX)


@dataclass
class Layer:
    model: str                  # 'hourly', 'daily', 'pinn' or 'ensemble'
    horizon_h: int
    p: np.ndarray               # fire probability per 1 km cell
    grid: Grid
    thresholds: dict
    burned: np.ndarray          # cells already burning (shown as ACTIVE_FIRE)
    issued: pd.Timestamp
    members: list
    extra: dict = field(default_factory=dict)    # extra per-cell values written to the geojson
    inputs: np.ndarray = None


def get_hourly_bundles():
    if not _HOURLY:
        for h in GOES_HORIZONS_H:
            if os.path.exists(os.path.join(BUNDLE_DIR, f'goes_h{h}', 'model.keras')):
                _HOURLY[h] = load_goes_bundle(h)
    return _HOURLY


def get_daily_bundle():
    if 'bundle' not in _DAILY:
        _DAILY['bundle'] = load_daily_bundle()
    return _DAILY['bundle']


def run_hourly(grid, t):
    bundles = get_hourly_bundles()
    if not bundles:
        print('no hourly models yet (train them in Section 8)')
        return []
    x, burned, out = goes_forecast(grid, t, bundles)
    return [Layer('hourly', h, p_u, grid, bundles[h]['thresholds'], burned, _utc(t).floor('h'), ['unet-goes'], inputs=x)
            for h, (p_u, _, _) in out.items()]


def run_daily(grid, t):
    bundle = get_daily_bundle()
    if bundle is None or not FIRMS_MAP_KEY or not EE_READY:
        print('daily model skipped: needs the saved model, a FIRMS key and Earth Engine')
        return []
    raw, burned = daily_inputs(grid, t)
    p = daily_predict(bundle, raw)
    return [Layer('daily', 24, p, grid, bundle['thresholds'], burned, _utc(t).floor('h'), ['unet-daily'],
                  inputs=raw['PrevFireMask'][..., None])]


def layer_geojson(layer):
    fmt = '%Y-%m-%dT%H:%M:%SZ'
    return to_geojson(layer.p, layer.grid, layer.issued.strftime(fmt), layer.thresholds, burned=layer.burned,
                      extra_layers=layer.extra,
                      collection_props={'valid_time': (layer.issued + pd.Timedelta(hours=layer.horizon_h)).strftime(fmt),
                                        'horizon_hours': layer.horizon_h, 'layer': 'spread_risk', 'model': layer.model,
                                        'members': layer.members, 'risk_thresholds': layer.thresholds})


def available_models():
    hourly = bool(get_hourly_bundles())
    return {'fire_check': True, 'current_fire': True, 'hourly': hourly, 'ensemble': hourly,
            'daily': bool(EE_READY and FIRMS_MAP_KEY and get_daily_bundle() is not None), 'pinn': False}


def serve_request(inst):
    """One request from the backend, e.g. {'model': 'hourly', 'lat': 34.07, 'lon': -118.58, 'time': 'now'}."""
    fmt = '%Y-%m-%dT%H:%M:%SZ'
    model = inst.get('model', 'hourly')
    when = inst.get('time', 'now')
    t = pd.Timestamp.now('UTC').floor('h') if when == 'now' else _utc(when).floor('h')
    if model == 'fire_check' and inst.get('lat') is None:
        fires = find_active_fires(goes_fire_points(t, bbox=CA_BBOX))
        return {'status': 'ok', 'time': t.strftime(fmt), 'fires': json.loads(fires.round(4).to_json(orient='records'))}
    lat, lon = float(inst['lat']), float(inst['lon'])
    check = fire_check(lat, lon, t, radius_km=float(inst.get('radius_km', FIRE_CHECK_RADIUS_KM)))
    base = {'time': t.strftime(fmt), 'model': model, 'active_fire': bool(check['active']),
            'nearest_fire_km': check['nearest_km']}
    if model == 'fire_check':
        return {'status': 'ok', **base, 'fires': json.loads(check['fires'].round(4).to_json(orient='records'))}
    if not check['active']:
        return {'status': 'no_active_fire', **base, 'layers': {}}
    if not available_models().get(model):
        return {'status': 'unavailable', **base, 'layers': {}, 'message': f'{model} is not enabled on this server'}
    layers = {}
    if model == 'current_fire':
        layers['current_fire'] = current_fire_geojson(Grid.around(lat, lon), t)
    elif model in ('hourly', 'ensemble'):          # ensemble = hourly U-Net until the PINN runs in the cloud
        for l in run_hourly(Grid.around(lat, lon), t):
            layers[f'h{l.horizon_h}'] = layer_geojson(l)
    elif model == 'daily':
        for l in run_daily(Grid.around(lat, lon, size=DAILY_TILE), t):
            layers['h24'] = layer_geojson(l)
    return {'status': 'ok', **base, 'layers': layers}


class ContractError(ValueError):
    """A request the service cannot answer; serve.py turns it into an HTTP error with a JSON message."""
    def __init__(self, message, status=400):
        super().__init__(message)
        self.status = status


def parse_issue_time(date_value, params):
    """'YYYY-MM-DD' -> 21:00 UTC (or additional_params.hour); a full ISO timestamp is used as given.
    Times in the future become the latest full hour. Returns (time, was_clamped)."""
    s = str(date_value or '').strip()
    if not s:
        raise ContractError('date is required, e.g. "2026-10-16" or "2026-10-16T21:00:00Z"')
    try:
        if len(s) <= 10:
            hour = int(params.get('hour', DEFAULT_HOUR_UTC))
            if not 0 <= hour <= 23:
                raise ContractError('additional_params.hour must be between 0 and 23')
            t = pd.Timestamp(s).tz_localize('UTC') + pd.Timedelta(hours=hour)
        else:
            t = _utc(s)
    except ContractError:
        raise
    except Exception:
        raise ContractError(f'could not read date {s!r}; use "YYYY-MM-DD" or an ISO-8601 UTC timestamp')
    t, now = t.floor('h'), pd.Timestamp.now('UTC').floor('h')
    return (now, True) if t > now else (t, False)


def parse_bbox(value):
    try:
        b = [float(v) for v in value]
    except Exception:
        raise ContractError('bbox must be [min_lon, min_lat, max_lon, max_lat] in degrees')
    if len(b) != 4 or not (-180 <= b[0] < b[2] <= 180 and -90 <= b[1] < b[3] <= 90):
        raise ContractError('bbox must be [min_lon, min_lat, max_lon, max_lat] with min < max')
    return tuple(b)


def predict_contract(req):
    """The backend contract: bbox + date + time_horizon -> one GeoJSON FeatureCollection with horizons 0/1/3/6."""
    model_type = str(req.get('model_type', 'unet')).lower()
    if model_type not in ('unet', 'ensemble'):
        raise ContractError(f"model_type '{model_type}' is not served by the U-Net service; "
                            "send it to the PINN service or let the ML adapter combine models")
    bbox = parse_bbox(req.get('bbox'))
    params = req.get('additional_params') or {}
    try:
        limit = int(req.get('time_horizon', 6))
        p_min = max(float(params.get('confidence_threshold', P_EMIT)), MIN_CONFIDENCE)
    except (TypeError, ValueError):
        raise ContractError('time_horizon must be a whole number of hours and confidence_threshold a number')
    horizons = [h for h in CONTRACT_HORIZONS if h <= limit]
    t, clamped = parse_issue_time(req.get('date'), params)
    try:
        grid = model_grid_for_bbox(bbox)
    except ValueError as ex:
        raise ContractError(str(ex))
    w, s, e, n = bbox
    radius = float(haversine_km(s, w, n, e)) / 2 + FIRE_SEARCH_PAD_KM
    check = fire_check((s + n) / 2, (w + e) / 2, t, radius_km=radius)
    meta = {'requested_model_type': model_type, 'time_clamped_to_latest_hour': clamped,
            'active_fires_nearby': int(len(check['fires'])), 'nearest_fire_km': check['nearest_km']}
    if not check['active'] or not horizons:
        return unified_feature_collection([], bbox, t, p_min, status='no_active_fire' if not check['active'] else 'ok',
                                          extra_metadata=meta)
    bundles = get_hourly_bundles()
    if not bundles:
        raise ContractError('the hourly models are not loaded on this server', status=503)
    x, burned, out = goes_forecast(grid, t, bundles)
    layers = []
    if 0 in horizons:      # current state: cells GOES-18 saw burning in the last hour
        layers.append({'h': 0, 'p': np.full(burned.shape, np.nan), 'grid': grid,
                       'thresholds': {'MODERATE_RISK': 1.0, 'HIGH_RISK': 1.0}, 'burned': x[..., 0] > 0})
    for h in horizons:
        if h in out:
            layers.append({'h': h, 'p': out[h][0], 'grid': grid, 'thresholds': bundles[h]['thresholds'], 'burned': burned})
    return unified_feature_collection(layers, bbox, t, p_min, extra_metadata=meta)

__all__ = [n for n in list(globals()) if not n.startswith('__')]   # lets the other modules import every name

In [ ]:
%%writefile {SERVING_DIR}/serve.py
"""U-Net wildfire spread service (FastAPI). Local: python serve.py --port 8081. Vertex AI: started by the Dockerfile."""
import argparse, os
from fastapi import FastAPI, Body
from fastapi.responses import JSONResponse
import inference

app = FastAPI(title='Fight Fire With AI - U-Net service')
HEALTH_ROUTE = os.environ.get('AIP_HEALTH_ROUTE', '/health')
PREDICT_ROUTE = os.environ.get('AIP_PREDICT_ROUTE', '/predict')


def _answer(inst):
    if 'bbox' in inst:                       # backend contract (bbox + date + time_horizon)
        return inference.predict_contract(inst)
    return inference.serve_request(inst)    # older lat/lon requests (Deploy & test examples)


@app.get(HEALTH_ROUTE)
def health():
    models = inference.available_models()
    return {'status': 'ok', 'models_loaded': bool(models.get('hourly')), 'models': models}


@app.post(PREDICT_ROUTE)
def predict(body: dict = Body(...)):
    try:
        if isinstance(body.get('instances'), list):          # Vertex AI wrapper in -> wrapper out
            params = body.get('parameters') or {}
            return {'predictions': [_answer({**params, **inst}) for inst in body['instances']]}
        return _answer(body)                                   # raw request -> bare FeatureCollection
    except inference.ContractError as ex:
        return JSONResponse(status_code=ex.status, content={'error': str(ex)})


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Run the U-Net service')
    parser.add_argument('--port', type=int, default=int(os.environ.get('AIP_HTTP_PORT', 8081)))
    parser.add_argument('--host', default='0.0.0.0')
    args = parser.parse_args()
    import uvicorn
    uvicorn.run(app, host=args.host, port=args.port)

In [ ]:
def build_static_cache(path=STATIC_CACHE, rows_per_call=218):
    """Terrain (elevation, slope) and the latest NDVI for all of California at 1 km, from Earth Engine. Only run once."""
    import ee
    import numpy as np, pandas as pd
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    x0, y0, res, full_h, full_w = -430_000.0, 480_000.0, 1000.0, 1090, 1000
    dem = ee.Image('USGS/SRTMGL1_003')
    today = ee.Date(pd.Timestamp.now('UTC').strftime('%Y-%m-%d'))
    ndvi = (ee.ImageCollection('NASA/VIIRS/002/VNP13A1').filterDate(today.advance(-40, 'day'), today)
              .sort('system:time_start', False).first())
    img = ee.Image.cat([dem.select('elevation'), ee.Terrain.slope(dem).rename('slope'),
                        ee.Image(ndvi).select('NDVI')]).toFloat()
    parts = {'elevation': [], 'slope': [], 'NDVI': []}
    for r0 in range(0, full_h, rows_per_call):
        n = min(rows_per_call, full_h - r0)
        arr = ee.data.computePixels({
            'expression': img, 'fileFormat': 'NUMPY_NDARRAY',
            'grid': {'dimensions': {'width': full_w, 'height': n},
                     'affineTransform': {'scaleX': res, 'shearX': 0, 'translateX': x0,
                                         'shearY': 0, 'scaleY': -res, 'translateY': y0 - r0 * res},
                     'crsCode': 'EPSG:3310'}})
        for k in parts:
            a = np.asarray(arr[k], np.float32)
            parts[k].append(np.where(np.isfinite(a), a, np.nan))
    np.savez_compressed(path, x0=x0, y0=y0, res=res, **{k: np.concatenate(v) for k, v in parts.items()})
    print('saved', path)


if BUILD_PACKAGE:
    import numpy as np, sklearn, keras
    import tensorflow as tf
    reqs = [f'tensorflow-cpu=={tf.__version__}', f'keras=={keras.__version__}', f'scikit-learn=={sklearn.__version__}',
            f'numpy=={np.__version__}', 'pandas', 'scipy', 'xarray', 'h5netcdf', 'pyproj', 'requests', 'joblib',
            'fastapi', 'uvicorn', 'earthengine-api', 'google-auth']
    open(os.path.join(SERVING_DIR, 'requirements.txt'), 'w').write('\n'.join(reqs) + '\n')
    open(os.path.join(SERVING_DIR, 'Dockerfile'), 'w').write('\n'.join([
        f'FROM python:{sys.version_info.major}.{sys.version_info.minor}-slim',
        'WORKDIR /app',
        'COPY requirements.txt .',
        'RUN pip install --no-cache-dir -r requirements.txt',
        'COPY . .',
        'ENV PYTHONUNBUFFERED=1',
        'EXPOSE 8081',
        'CMD exec uvicorn serve:app --host 0.0.0.0 --port ${AIP_HTTP_PORT:-8081}']) + '\n')

    mdir = os.path.join(SERVING_DIR, 'models')
    shutil.rmtree(mdir, ignore_errors=True)
    os.makedirs(mdir)
    for f in ['model.keras', 'features.json', 'stats.json', 'calibrator.joblib', 'thresholds.json', 'manifest.json']:
        if os.path.exists(os.path.join(BUNDLE_DIR, f)):
            shutil.copy(os.path.join(BUNDLE_DIR, f), mdir)
    for h in (1, 3, 6):
        src = os.path.join(BUNDLE_DIR, f'goes_h{h}')
        assert os.path.isdir(src), f'missing {src}: train the hourly models in the final notebook first'
        shutil.copytree(src, os.path.join(mdir, f'goes_h{h}'))

    if not os.path.exists(STATIC_CACHE):
        build_static_cache()
    shutil.copy(STATIC_CACHE, os.path.join(SERVING_DIR, 'static_ca_1km.npz'))
    size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(SERVING_DIR) for f in fs)
    print(f'package ready in {SERVING_DIR} ({size / 1e6:.0f} MB):', sorted(os.listdir(SERVING_DIR)))

## 4. Test it here first

This calls the package exactly the way the backend and Vertex AI will, but on this computer, with the backend's request
format (a bbox, a date and `time_horizon`). You should see `status ok` and cells for horizons 0, 1, 3 and 6. The first
forecast downloads a day of satellite files, so it can take a minute or two.

In [ ]:
if LOCAL_TEST:
    for pkg in ('fastapi', 'httpx', 'uvicorn'):
        if importlib.util.find_spec(pkg) is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    os.environ['WILDFIRE_MODEL_DIR'] = os.path.join(SERVING_DIR, 'models')
    os.environ['WILDFIRE_STATIC_CACHE'] = os.path.join(SERVING_DIR, 'static_ca_1km.npz')
    os.environ['WILDFIRE_WARM_CACHE'] = '0'          # the background download only runs in the cloud
    if SERVING_DIR not in sys.path:
        sys.path.insert(0, SERVING_DIR)
    for _m in ('settings', 'postprocessing', 'native_ingestion', 'inference', 'serve'):   # reload in order
        if _m in sys.modules:
            importlib.reload(sys.modules[_m])
        else:
            importlib.import_module(_m)
    import serve
    from fastapi.testclient import TestClient
    client = TestClient(serve.app)

    print('health:', client.get('/health').json())
    request = {'bbox': [-118.75, 33.95, -118.35, 34.25], 'time_horizon': 6, 'model_type': 'unet', 'date': '2025-01-10'}
    t0 = time.time()
    r = client.post('/predict', json=request)
    fc = r.json()
    per_h = {h: sum(f['properties']['horizon_hours'] == h for f in fc['features']) for h in (0, 1, 3, 6)}
    print(f"contract request: status {fc['metadata']['status']} | cells per horizon {per_h} | "
          f"{len(r.content) / 1024:.0f} KB | {time.time() - t0:.0f} s")

## 5. Deploy next to the PINN model

This step:

1. builds the container in the cloud with Cloud Build (no Docker needed here),
2. stores it in Artifact Registry,
3. adds it to Model Registry as `wildfire-unet`,
4. deploys it on the endpoint `wildfire-unet-endpoint` (created the first time).

Running it again later, for example after retraining, swaps the new model onto the **same** endpoint, takes the old one
off and deletes it. The endpoint ID the backend uses stays the same. It takes about 20-40 minutes. Set `DEPLOY = True`
in step 1 first.

In [ ]:
def _ids(text):
    return [t.split('/')[-1] for t in re.split(r'[;\s]+', text) if t.strip()]


def find_ids(kind, name):
    return _ids(sh(f'gcloud ai {kind} list --region={GCP_REGION} --filter=display_name={name} '
                   f'--sort-by=~createTime --format="value(name)"', check=False))


def deployed_ids(endpoint_id):
    return _ids(sh(f'gcloud ai endpoints describe {endpoint_id} --region={GCP_REGION} '
                   f'--format="value(deployedModels[].id)"', check=False))


if DEPLOY:
    sh(f'gcloud config set project {GCP_PROJECT}')
    sh('gcloud services enable aiplatform.googleapis.com artifactregistry.googleapis.com cloudbuild.googleapis.com')
    sh(f'gcloud artifacts repositories create {AR_REPO} --repository-format=docker --location={GCP_REGION}', check=False)
    image = f'{IMAGE_BASE}:{time.strftime("%Y%m%d-%H%M%S")}'      # a new tag for every build
    sh(f'gcloud builds submit {SERVING_DIR} --tag {image} --timeout=3600s')

    old_models = find_ids('models', MODEL_NAME)
    sh(f'gcloud ai models upload --region={GCP_REGION} --display-name={MODEL_NAME} --container-image-uri={image} '
       f'--container-predict-route=/predict --container-health-route=/health --container-ports=8080 '
       f'--container-env-vars=GCP_PROJECT={GCP_PROJECT}' + (f',FIRMS_MAP_KEY={FIRMS_MAP_KEY}' if FIRMS_MAP_KEY else ''))
    model_id = [m for m in find_ids('models', MODEL_NAME) if m not in old_models][0]

    endpoints = find_ids('endpoints', ENDPOINT_NAME)
    if not endpoints:
        sh(f'gcloud ai endpoints create --region={GCP_REGION} --display-name={ENDPOINT_NAME}')
        endpoints = find_ids('endpoints', ENDPOINT_NAME)
    endpoint_id = endpoints[0]
    previous = deployed_ids(endpoint_id)

    sh(f'gcloud ai endpoints deploy-model {endpoint_id} --region={GCP_REGION} --model={model_id} '
       f'--display-name={MODEL_NAME} --machine-type={MACHINE_TYPE} --min-replica-count=1 --max-replica-count=1 '
       f'--traffic-split=0=100')
    for dep in previous:                       # take the old version off so only one machine is billed
        sh(f'gcloud ai endpoints undeploy-model {endpoint_id} --region={GCP_REGION} --deployed-model-id={dep} --quiet')
    for old in old_models:                     # remove old versions from Model Registry
        sh(f'gcloud ai models delete {old} --region={GCP_REGION} --quiet', check=False)

    json.dump({'project': GCP_PROJECT, 'region': GCP_REGION, 'model_id': model_id, 'endpoint_id': endpoint_id},
              open(ENDPOINT_FILE, 'w'), indent=1)
    print(f'deployed: model {model_id} on endpoint {endpoint_id} (saved to {ENDPOINT_FILE})')

## 6. Test it in the cloud

Either run the cell below, or do it in the console like the PINN guide: **Agent Platform → Models → Model Registry →
wildfire-unet → Deploy & test**, paste one of these and click **Infer**:

```json
{"instances": [{"model": "fire_check", "time": "2025-01-10T21:00:00Z"}]}
```

```json
{"instances": [{"model": "hourly", "lat": 34.074, "lon": -118.559, "time": "2025-01-10T21:00:00Z"}]}
```

A good answer has `"status": "ok"` and, for `hourly`, `layers` with `h1`, `h3` and `h6` (each a GeoJSON map). Live
requests (`"time": "now"`) are fastest because the server keeps the last day of satellite files downloaded. A replay of
an old date downloads its files on the first request; if that one times out, click **Infer** again a minute later.

In [ ]:
def call_endpoint(instance, timeout=300):
    """Calls the deployed model the same way the backend will (REST with a Google access token)."""
    import requests
    ids = json.load(open(ENDPOINT_FILE))
    token = sh('gcloud auth print-access-token')
    url = (f"https://{ids['region']}-aiplatform.googleapis.com/v1/projects/{ids['project']}"
           f"/locations/{ids['region']}/endpoints/{ids['endpoint_id']}:predict")
    r = requests.post(url, headers={'Authorization': f'Bearer {token}'}, json={'instances': [instance]}, timeout=timeout)
    r.raise_for_status()
    return r.json()['predictions'][0]


if os.path.exists(ENDPOINT_FILE):
    for request in [{'model': 'fire_check', 'time': '2025-01-10T21:00:00Z'},
                    {'model': 'hourly', 'lat': 34.074, 'lon': -118.559, 'time': '2025-01-10T21:00:00Z'}]:
        try:
            pred = call_endpoint(request)
            print(request['model'], '->', pred['status'], {k: len(v['features']) for k, v in pred.get('layers', {}).items()})
        except Exception as ex:
            print(request['model'], '-> failed:', str(ex)[:200], '(if it timed out, run this cell again in a minute)')

## 7. Stop paying when you're not using it

Undeploying takes the model off the machine, so the hourly cost stops. The model stays in Model Registry; deploy it again
from its **Deploy & test** tab, or by running step 5 again.

In [ ]:
def stop_endpoint():
    """Takes every model off the endpoint; the endpoint and the model in Model Registry are kept."""
    ids = json.load(open(ENDPOINT_FILE))
    for dep in _ids(sh(f"gcloud ai endpoints describe {ids['endpoint_id']} --region={ids['region']} "
                       f"--format=\"value(deployedModels[].id)\"")):
        sh(f"gcloud ai endpoints undeploy-model {ids['endpoint_id']} --region={ids['region']} --deployed-model-id={dep} --quiet")
    print('undeployed; the endpoint and model are kept')


STOP_NOW = False
if STOP_NOW:
    stop_endpoint()

## For the backend team

Send one or more requests in `instances`:

| Field | Values |
|---|---|
| `model` | `fire_check`, `current_fire`, `hourly`, `daily` or `ensemble` |
| `lat`, `lon` | location of the fire (leave out for `fire_check` to scan all of California) |
| `time` | `"now"` or a UTC time such as `"2025-01-10T21:00:00Z"` |

Each answer has `status` (`ok`, `no_active_fire` or `unavailable`) and `layers`: one GeoJSON map per forecast
(`h1`, `h3`, `h6` for hourly, `h24` for daily). Callers need the Vertex AI User role on `wildfirepinn`, the same as for the
PINN model.

Note: `ensemble` currently returns the hourly U-Net only. `daily` stays `unavailable` until two things are set: the
FIRMS key (put `FIRMS_MAP_KEY` in Colab Secrets before step 5) and Earth Engine access for the endpoint's service account
(ask the project owner).